# Практика: перегрузка операторов

Один из вариантов реализации [полиморфизма](conspects/04-polymorphism.md) для пользовательских классов — **перегрузка операторов и встроенных функций**: класс определяет специальные методы, и после этого операторы и встроенные функции Python работают с его экземплярами.

На этом занятии:

- определим строковые представления экземпляра методами `__repr__` и `__str__`;
- зададим логическое значение экземпляра методом `__bool__`;
- перегрузим операторы сравнения `==`, `!=`, `>`, `<`, `>=` и `<=`;
- перегрузим аддитивные операции `+` и `-`;
- разберем, как интерпретатор выполняет бинарную операцию, и познакомимся с отраженными операциями и объектом `NotImplemented`;
- перегрузим мультипликативные операции `*` и `/`.

Все темы разбираем на одном примере: класс `Square`, который описывает квадрат.

In [ ]:
import math

## 1. Строковые представления: `__repr__` и `__str__`

У класса `Square` один служебный атрибут `_side` — длина стороны квадрата. Конструктор принимает длину стороны, проверяет, что она не отрицательна, и сохраняет ее в атрибут `_side`.

In [ ]:
class Square:
    def __init__(self, side: float) -> None:
        if side < 0:
            raise ValueError(f"side must not be negative: {side}")

        self._side = side

Как мы видели на прошлом занятии, по умолчанию строковое представление экземпляра пользовательского класса содержит только имя класса и адрес объекта в памяти. Значения атрибутов из него узнать нельзя.

In [ ]:
square = Square(side=5)
print(f"repr: {square!r}")  # repr: <__main__.Square object at 0x...>
print(f"str: {square!s}")   # str: <__main__.Square object at 0x...>

Понятное строковое представление умеет создавать декоратор `dataclass`, но формат этого представления задан декоратором. Чтобы управлять форматом самостоятельно, в классе определяют специальные методы:

| Метод | Кто вызывает | Назначение |
|---|---|---|
| `__repr__` | `repr()`, формат `!r`, вывод значения в интерактивной оболочке | представление для разработчика: однозначное, по возможности похожее на выражение, которое создает такой же объект |
| `__str__` | `str()`, `print()`, формат `!s`, f-строки без спецификатора | представление для пользователя программы: удобное для чтения |

Оба метода принимают только `self` и обязаны возвращать строку. Вызов `print` внутри них добавлен, чтобы увидеть, в какой момент вызывается каждый метод.

In [ ]:
class Square:
    def __init__(self, side: float) -> None:
        if side < 0:
            raise ValueError(f"side must not be negative: {side}")

        self._side = side

    def __str__(self) -> str:
        print("call __str__")
        return f"square with side {self._side}"

    def __repr__(self) -> str:
        print("call __repr__")
        return f"Square(side={self._side})"

In [ ]:
square = Square(side=5)
print(f"repr: {square!r}")
print(f"str: {square!s}")

Мы не вызывали `__repr__` и `__str__` сами: это сделал интерпретатор при вычислении f-строки. Специальные методы предназначены именно для такого **неявного** вызова. Явная запись `square.__repr__()` допустима с точки зрения языка, но так писать не принято: вместо нее используют `repr(square)`.

Элементы контейнеров всегда выводятся с помощью `__repr__`, даже если сам контейнер печатается функцией `print`:

In [ ]:
print([Square(1), Square(2)])

Если оба представления должны совпадать, достаточно определить только `__repr__`. Реализация `__str__`, унаследованная от `object`, вызывает `__repr__`. Обратное неверно: если определить только `__str__`, функция `repr()` по-прежнему вернет строку с адресом объекта.

In [ ]:
class Square:
    def __init__(self, side: float) -> None:
        if side < 0:
            raise ValueError(f"side must not be negative: {side}")

        self._side = side

    def __repr__(self) -> str:
        return f"Square(side={self._side})"

In [ ]:
square = Square(side=5)
print(f"repr: {square!r}")  # repr: Square(side=5)
print(f"str: {square!s}")   # str: Square(side=5)

## 2. Логическое значение: `__bool__`

Любой объект можно проверить на истинность: в условии `if` или функцией `bool()`. По умолчанию экземпляры пользовательских классов считаются истинными. Чтобы изменить это правило, определяют метод `__bool__`: он принимает только `self` и обязан вернуть `True` или `False`.

Будем считать квадрат истинным, если его площадь не равна нулю. Площадь — число с плавающей точкой, поэтому для сравнения используем функцию `math.isclose`. Ее параметр `rel_tol` задает **относительную** погрешность: она вычисляется как доля большего из сравниваемых чисел. При сравнении с нулем такая погрешность бесполезна: с настройками по умолчанию нулю «близок» только сам ноль. Поэтому для сравнения с нулем обязательно задают **абсолютную** погрешность `abs_tol`.

In [ ]:
print(math.isclose(1e-30, 0))                # False
print(math.isclose(1e-30, 0, abs_tol=1e-9))  # True

In [ ]:
class Square:
    def __init__(self, side: float) -> None:
        if side < 0:
            raise ValueError(f"side must not be negative: {side}")

        self._side = side

    def __repr__(self) -> str:
        return f"Square(side={self._side})"

    def __bool__(self) -> bool:
        return not math.isclose(self.get_area(), 0, abs_tol=1e-9)

    def get_area(self) -> float:
        return self._side ** 2

In [ ]:
print(bool(Square(side=5)))  # True
print(bool(Square(side=0)))  # False

if not Square(side=0):
    print("The square is empty")

## 3. Операции сравнения: `__eq__` и `__ne__`

Перегрузим операторы `==` и `!=`. Будем считать, что два квадрата равны, если равны длины их сторон. Стороны — числа с плавающей точкой, поэтому снова сравниваем их с помощью `math.isclose`.

За оператор `==` отвечает метод `__eq__` (от *equal*), за `!=` — метод `__ne__` (от *not equal*). Оба принимают `self` и второй операнд. Обычно они возвращают `bool`: язык разрешает вернуть объект другого типа, но без веской причины делать этого не следует.

Имя `Square` в аннотациях параметров записано строкой, т.к. во время выполнения тела класса объект класса `Square` еще не создан.

In [ ]:
class Square:
    def __init__(self, side: float) -> None:
        if side < 0:
            raise ValueError(f"side must not be negative: {side}")

        self._side = side

    def __repr__(self) -> str:
        return f"Square(side={self._side})"

    def __bool__(self) -> bool:
        return not math.isclose(self.get_area(), 0, abs_tol=1e-9)

    def __eq__(self, other: "Square") -> bool:
        return math.isclose(self._side, other._side)

    def __ne__(self, other: "Square") -> bool:
        return not (self == other)

    def get_area(self) -> float:
        return self._side**2

In [ ]:
first_square = Square(5)
second_square = Square(3.14)
print(first_square == second_square)     # False
print(first_square != second_square)     # True
print(Square(0.1 + 0.2) == Square(0.3))  # True

Если метод `__ne__` только возвращает значение, противоположное результату `__eq__`, его явное определение можно опустить. Причина в том, что реализация `__ne__`, унаследованная от `object`, делает именно это: вызывает `__eq__` и возвращает противоположное значение. Поэтому достаточно определить только `__eq__`:

In [ ]:
class Square:
    def __init__(self, side: float) -> None:
        if side < 0:
            raise ValueError(f"side must not be negative: {side}")

        self._side = side

    def __repr__(self) -> str:
        return f"Square(side={self._side})"

    def __bool__(self) -> bool:
        return not math.isclose(self.get_area(), 0, abs_tol=1e-9)

    def __eq__(self, other: "Square") -> bool:
        print("call __eq__")
        return math.isclose(self._side, other._side)

    def get_area(self) -> float:
        return self._side**2

In [ ]:
first_square = Square(5)
second_square = Square(3.14)
print(first_square == second_square)
print(first_square != second_square)

Оба оператора вызвали метод `__eq__`.

У перегрузки `==` есть важное следствие: если класс определяет `__eq__` и не определяет `__hash__`, Python устанавливает атрибуту `__hash__` значение `None`, и экземпляры становятся **нехешируемыми** (см. [конспект о хешах](../lesson06/conspects/02-hash-and-hashability.md)). Причина — требование: у равных объектов должны быть равные хеши, а унаследованный от `object` хеш этому требованию уже не удовлетворяет.

In [ ]:
try:
    hash(Square(5))

except TypeError as error:
    print(error)

## 4. Отношения порядка: `__gt__` и `__lt__`

Определим для квадратов отношение порядка: один квадрат больше другого, если его площадь больше. Стороны не отрицательны, поэтому для этого достаточно сравнить длины сторон.

Операторам строгого сравнения соответствуют методы `__gt__` (от *greater than*) и `__lt__` (от *less than*). Они принимают те же аргументы, что и `__eq__`. Квадраты, равные с точки зрения `__eq__`, не должны быть больше или меньше друг друга, поэтому сначала проверяем, что квадраты не равны.

In [ ]:
class Square:
    def __init__(self, side: float) -> None:
        if side < 0:
            raise ValueError(f"side must not be negative: {side}")

        self._side = side

    def __repr__(self) -> str:
        return f"Square(side={self._side})"

    def __bool__(self) -> bool:
        return not math.isclose(self.get_area(), 0, abs_tol=1e-9)

    def __eq__(self, other: "Square") -> bool:
        return math.isclose(self._side, other._side)

    def __gt__(self, other: "Square") -> bool:
        return self != other and self._side > other._side

    def __lt__(self, other: "Square") -> bool:
        return self != other and self._side < other._side

    def get_area(self) -> float:
        return self._side**2

In [ ]:
first_square = Square(5)
second_square = Square(3.14)
print(first_square > second_square)
print(first_square < second_square)

Перегрузка операторов сравнения позволяет передавать экземпляры класса во встроенные функции, которые сравнивают элементы. Функции `sorted` и `min`, а также метод списка `sort` используют только оператор `<`, то есть метод `__lt__`. Функция `max` использует оператор `>`, то есть метод `__gt__`.

In [ ]:
squares = [Square(5), Square(1), Square(3)]
print(sorted(squares))
print(max(squares))

## 5. Практическая задача: нестрогое сравнение

### Условие

Операторам нестрогого сравнения `>=` и `<=` соответствуют методы `__ge__` (от *greater than or equal*) и `__le__` (от *less than or equal*). Сейчас для квадратов эти операторы не определены.

Реализуйте методы `__ge__` и `__le__` в классе `OrderedSquare`. Он [наследуется](conspects/05-inheritance.md) от `Square`, поэтому методы `__eq__`, `__gt__` и `__lt__` в нем уже есть.

Требования:

- `first >= second` возвращает `True`, если квадрат `first` больше квадрата `second` или равен ему;
- `first <= second` возвращает `True`, если квадрат `first` меньше квадрата `second` или равен ему;
- равенство понимается так же, как в методе `__eq__`, то есть с учетом погрешности вычислений с плавающей точкой.

### Пример

```python
print(OrderedSquare(5) >= OrderedSquare(3))            # True
print(OrderedSquare(5) <= OrderedSquare(3))            # False
print(OrderedSquare(0.1 + 0.2) <= OrderedSquare(0.3))  # True
```

In [ ]:
class OrderedSquare(Square):
    def __ge__(self, other: "Square") -> bool:
        # ваш код
        return False

    def __le__(self, other: "Square") -> bool:
        # ваш код
        return False

### Проверка

In [ ]:
assert OrderedSquare(5) >= OrderedSquare(3)
assert not OrderedSquare(5) <= OrderedSquare(3)
assert OrderedSquare(3) <= OrderedSquare(5)
assert not OrderedSquare(3) >= OrderedSquare(5)
assert OrderedSquare(4) >= OrderedSquare(4)
assert OrderedSquare(4) <= OrderedSquare(4)
assert OrderedSquare(0) <= OrderedSquare(0)
assert OrderedSquare(0.1 + 0.2) <= OrderedSquare(0.3)
assert OrderedSquare(0.3) >= OrderedSquare(0.1 + 0.2)

print("All tests passed")

Если в классе определены `__eq__` и один из методов `__lt__`, `__le__`, `__gt__` или `__ge__`, остальные методы сравнения можно не писать вручную, а добавить с помощью декоратора класса [`functools.total_ordering`](https://docs.python.org/3/library/functools.html#functools.total_ordering).

## 6. Аддитивные операции

### Прямые операции

Результатом сложения двух квадратов будем считать новый квадрат, площадь которого равна сумме площадей исходных. Вычитание определим аналогично: площадь результата равна разности площадей. Обе операции возвращают **новый** объект и не изменяют операнды.

Операторам `+` и `-` соответствуют методы `__add__` и `__sub__`. Вычесть из меньшего квадрата больший нельзя, поскольку площадь не может быть отрицательной, поэтому в этом случае возбуждаем `ValueError`.

Из-за погрешности вычислений у двух равных (с точки зрения `__eq__`) квадратов разность площадей может оказаться очень маленьким отрицательным числом. Возведение такого числа в степень `0.5` дает комплексное число, поэтому разность площадей ограничиваем снизу нулем с помощью `max`.

In [ ]:
class Square:
    def __init__(self, side: float) -> None:
        if side < 0:
            raise ValueError(f"side must not be negative: {side}")

        self._side = side

    def __repr__(self) -> str:
        return f"Square(side={self._side})"

    def __bool__(self) -> bool:
        return not math.isclose(self.get_area(), 0, abs_tol=1e-9)

    def __eq__(self, other: "Square") -> bool:
        return math.isclose(self._side, other._side)

    def __gt__(self, other: "Square") -> bool:
        return self != other and self._side > other._side

    def __lt__(self, other: "Square") -> bool:
        return self != other and self._side < other._side

    def __add__(self, other: "Square") -> "Square":
        common_area = self.get_area() + other.get_area()
        return Square(side=common_area**0.5)

    def __sub__(self, other: "Square") -> "Square":
        if self < other:
            raise ValueError(f"cannot subtract {other} from {self}")

        common_area = max(self.get_area() - other.get_area(), 0)
        return Square(side=common_area**0.5)

    def get_area(self) -> float:
        return self._side**2

In [ ]:
first_square = Square(3)
second_square = Square(4)
print(first_square + second_square)  # Square(side=5.0)
print(second_square - first_square)  # Square(side=2.6457513110645907)

try:
    first_square - second_square

except ValueError as error:
    print(error)

### Операции с числами

Разрешим складывать квадрат с числом и вычитать из него число. Число будем считать квадратом, длина стороны которого равна этому числу.

Числом будем считать экземпляр типа `int` или `float`. В аннотациях достаточно указать `float`: по [правилам аннотаций типов](https://typing.python.org/en/latest/spec/special-types.html#special-cases-for-float-and-complex) там, где ожидается `float`, допустимо и значение типа `int`. На функцию `isinstance` это правило не распространяется: `isinstance(4, float)` возвращает `False`, поэтому при проверке указываем оба типа: `isinstance(value, int | float)`.

In [ ]:
class Square:
    def __init__(self, side: float) -> None:
        if side < 0:
            raise ValueError(f"side must not be negative: {side}")

        self._side = side

    def __repr__(self) -> str:
        return f"Square(side={self._side})"

    def __bool__(self) -> bool:
        return not math.isclose(self.get_area(), 0, abs_tol=1e-9)

    def __eq__(self, other: "Square") -> bool:
        return math.isclose(self._side, other._side)

    def __gt__(self, other: "Square") -> bool:
        return self != other and self._side > other._side

    def __lt__(self, other: "Square") -> bool:
        return self != other and self._side < other._side

    def __add__(self, other: "Square | float") -> "Square":
        if isinstance(other, int | float):
            other = Square(side=other)

        common_area = self.get_area() + other.get_area()
        return Square(side=common_area**0.5)

    def __sub__(self, other: "Square | float") -> "Square":
        if isinstance(other, int | float):
            other = Square(side=other)

        if self < other:
            raise ValueError(f"cannot subtract {other} from {self}")

        common_area = max(self.get_area() - other.get_area(), 0)
        return Square(side=common_area**0.5)

    def get_area(self) -> float:
        return self._side**2

In [ ]:
first_square = Square(3)
second_square = Square(4)
print(first_square + 4)   # Square(side=5.0)
print(second_square - 3)  # Square(side=2.6457513110645907)

### Отраженные операции

У текущей реализации есть недостаток: выражение `Square(3) + 4` работает, а `4 + Square(3)` — нет.

In [ ]:
try:
    4 + Square(3)

except TypeError as error:
    print(error)

Чтобы понять причину, разберем, как интерпретатор вычисляет выражение `left + right`:

1. Если в классе левого операнда есть метод `__add__`, вызывается `left.__add__(right)`.
2. Если этот метод вернул любой объект, кроме `NotImplemented`, он становится результатом операции.
3. Если метода `__add__` нет или он вернул `NotImplemented`, интерпретатор ищет в классе правого операнда **отраженный** метод `__radd__` и вызывает `right.__radd__(left)`.
4. Если метода `__radd__` нет или он тоже вернул `NotImplemented`, возбуждается `TypeError`.

![Порядок вычисления бинарной операции left + right](images/binary_operator.svg)

`NotImplemented` — специальный объект, существующий в единственном экземпляре. Возвращая его, метод сообщает интерпретатору, что не поддерживает операцию с таким операндом. Его не следует путать с исключением `NotImplementedError`: объект `NotImplemented` **возвращают** инструкцией `return`.

У этого алгоритма есть два уточнения:

- если операнды имеют один и тот же тип, отраженный метод не вызывается;
- если тип правого операнда является дочерним классом типа левого и по-своему реализует отраженный метод, то `right.__radd__` вызывается **первым**. Благодаря этому дочерний класс может изменить поведение операции.

В выражении `4 + Square(3)` метод `int.__add__` не знает о классе `Square` и возвращает `NotImplemented`, а метода `__radd__` в классе `Square` нет. Остальные бинарные арифметические операции выполняются по тем же правилам: имя отраженного метода получается добавлением буквы `r` (от *reflected*): `__rsub__`, `__rmul__`, `__rtruediv__`.

Добавим в класс метод `__radd__`. Кроме того, исправим прямые методы: для операнда неподдерживаемого типа они должны возвращать `NotImplemented`. Так класс другого операнда получает возможность выполнить операцию самостоятельно. То же правило действует для методов сравнения: сейчас выражение `Square(1) == "text"` завершается ошибкой `AttributeError`, хотя должно возвращать `False`.

У операторов сравнения нет отдельных отраженных методов: для `__lt__` отраженным является `__gt__`, для `__le__` — `__ge__`, а `__eq__` и `__ne__` отражают сами себя.

Если и прямой, и отраженный методы сравнения вернули `NotImplemented`, дальнейшее поведение зависит от оператора:

- операторы `==` и `!=` проверяют идентичность объектов: `left == right` дает тот же результат, что и `left is right`, а `left != right` — тот же, что и `left is not right`;
- операторы `<`, `<=`, `>` и `>=` возбуждают `TypeError`.

In [ ]:
class Square:
    def __init__(self, side: float) -> None:
        if side < 0:
            raise ValueError(f"side must not be negative: {side}")

        self._side = side

    def __repr__(self) -> str:
        return f"Square(side={self._side})"

    def __bool__(self) -> bool:
        return not math.isclose(self.get_area(), 0, abs_tol=1e-9)

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Square):
            return NotImplemented

        return math.isclose(self._side, other._side)

    def __gt__(self, other: "Square") -> bool:
        if not isinstance(other, Square):
            return NotImplemented

        return self != other and self._side > other._side

    def __lt__(self, other: "Square") -> bool:
        if not isinstance(other, Square):
            return NotImplemented

        return self != other and self._side < other._side

    def __add__(self, other: "Square | float") -> "Square":
        if not isinstance(other, Square | int | float):
            return NotImplemented

        if isinstance(other, int | float):
            other = Square(side=other)

        common_area = self.get_area() + other.get_area()
        return Square(side=common_area**0.5)

    def __radd__(self, other: float) -> "Square":
        return self + other

    def __sub__(self, other: "Square | float") -> "Square":
        if not isinstance(other, Square | int | float):
            return NotImplemented

        if isinstance(other, int | float):
            other = Square(side=other)

        if self < other:
            raise ValueError(f"cannot subtract {other} from {self}")

        common_area = max(self.get_area() - other.get_area(), 0)
        return Square(side=common_area**0.5)

    def get_area(self) -> float:
        return self._side**2

In [ ]:
print(Square(3) + 4)
print(4 + Square(3))
print(Square(1) == "text")

try:
    Square(3) + None

except TypeError as error:
    print(error)

Функция `sum` начинает сложение с числа `0`, поэтому первой операцией оказывается `0 + Square(...)`. Теперь она выполняется с помощью `__radd__`, и квадраты можно складывать функцией `sum`:

In [ ]:
print(sum([Square(3), Square(4)]))

## 7. Мультипликативные операции

Умножение квадрата на число создает новый квадрат, площадь которого больше площади исходного в указанное число раз. Деление на число создает квадрат, площадь которого в указанное число раз меньше.

Операторам `*` и `/` соответствуют методы `__mul__` и `__truediv__`. Отрицательный множитель дал бы отрицательную площадь, поэтому в этом случае возбуждаем `ValueError`. Деление выражаем через уже реализованное умножение; при делении на ноль выражение `1 / divisor` возбуждает `ZeroDivisionError`.

In [ ]:
class Square:
    def __init__(self, side: float) -> None:
        if side < 0:
            raise ValueError(f"side must not be negative: {side}")

        self._side = side

    def __repr__(self) -> str:
        return f"Square(side={self._side})"

    def __bool__(self) -> bool:
        return not math.isclose(self.get_area(), 0, abs_tol=1e-9)

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Square):
            return NotImplemented

        return math.isclose(self._side, other._side)

    def __gt__(self, other: "Square") -> bool:
        if not isinstance(other, Square):
            return NotImplemented

        return self != other and self._side > other._side

    def __lt__(self, other: "Square") -> bool:
        if not isinstance(other, Square):
            return NotImplemented

        return self != other and self._side < other._side

    def __add__(self, other: "Square | float") -> "Square":
        if not isinstance(other, Square | int | float):
            return NotImplemented

        if isinstance(other, int | float):
            other = Square(side=other)

        common_area = self.get_area() + other.get_area()
        return Square(side=common_area**0.5)

    def __radd__(self, other: float) -> "Square":
        return self + other

    def __sub__(self, other: "Square | float") -> "Square":
        if not isinstance(other, Square | int | float):
            return NotImplemented

        if isinstance(other, int | float):
            other = Square(side=other)

        if self < other:
            raise ValueError(f"cannot subtract {other} from {self}")
        common_area = max(self.get_area() - other.get_area(), 0)
        return Square(side=common_area**0.5)

    def __mul__(self, factor: float) -> "Square":
        if not isinstance(factor, int | float):
            return NotImplemented

        if factor < 0:
            raise ValueError(f"factor must not be negative: {factor}")

        scaled_area = self.get_area() * factor
        return Square(side=scaled_area**0.5)

    def __truediv__(self, divisor: float) -> "Square":
        if not isinstance(divisor, int | float):
            return NotImplemented

        return self * (1 / divisor)

    def get_area(self) -> float:
        return self._side**2

In [ ]:
square = Square(2)
print(square * 4)  # Square(side=4.0)
print(square / 4)  # Square(side=1.0)

try:
    square / 0

except ZeroDivisionError as error:
    print(error)

## 8. Практическая задача: отраженные умножение и вычитание

### Условие

Сейчас выражения `4 * Square(2)` и `10 - Square(6)` завершаются ошибкой `TypeError`. Реализуйте в классе `ScalableSquare`, дочернем классе `Square`, отраженные методы `__rmul__` и `__rsub__`.

Требования:

- `number * square` дает тот же результат, что и `square * number`;
- `number - square` вычисляется так: число считается квадратом с длиной стороны `number`, и из него вычитается `square`;
- если площадь вычитаемого квадрата больше, возбуждается `ValueError`;
- если левый операнд не является экземпляром `int` или `float`, оба метода возвращают `NotImplemented`.

Отраженный метод для деления не нужен: операция «число разделить на квадрат» в нашей задаче не имеет смысла.

### Пример

```python
print(4 * ScalableSquare(2))   # Square(side=4.0)
print(10 - ScalableSquare(6))  # Square(side=8.0)
```

In [ ]:
class ScalableSquare(Square):
    def __rmul__(self, other: float) -> "Square":
        # ваш код
        return self

    def __rsub__(self, other: float) -> "Square":
        # ваш код
        return self

### Проверка

In [ ]:
assert 4 * ScalableSquare(2) == Square(4)
assert ScalableSquare(2) * 4 == Square(4)
assert 0.25 * ScalableSquare(2) == Square(1)
assert 0 * ScalableSquare(2) == Square(0)

assert 10 - ScalableSquare(6) == Square(8)
assert 5.0 - ScalableSquare(3) == Square(4)
assert ScalableSquare(10) - 6 == Square(8)
assert 3 - ScalableSquare(3) == Square(0)

try:
    3 - ScalableSquare(4)

except ValueError:
    pass

else:
    raise AssertionError("ValueError was not raised for a larger subtrahend")

try:
    -1 * ScalableSquare(2)

except ValueError:
    pass

else:
    raise AssertionError("ValueError was not raised for a negative factor")

assert ScalableSquare(2).__rmul__("text") is NotImplemented
assert ScalableSquare(2).__rsub__("text") is NotImplemented

print("All tests passed")

## Материалы

Мы рассмотрели не все специальные методы. Кроме них есть методы для остальных арифметических операций (`__floordiv__`, `__mod__`, `__pow__`), унарных операций (`__neg__`, `__abs__`), составного присваивания (`__iadd__`), а также для работы объекта как контейнера (`__len__`, `__getitem__`, `__contains__`).

- [Python documentation. Data model — Special method names](https://docs.python.org/3/reference/datamodel.html#special-method-names) — полный список специальных методов;
- [Python documentation. Data model — Emulating numeric types](https://docs.python.org/3/reference/datamodel.html#emulating-numeric-types) — арифметические операции;
- [Python documentation. The numbers module — Implementing the arithmetic operations](https://docs.python.org/3/library/numbers.html#implementing-the-arithmetic-operations) — рекомендации по реализации арифметических операций.